In [1]:
# Import required libraries
import warnings
warnings.filterwarnings("ignore")

import sqlite3
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [2]:
DB_PATH = "summarize_middleware_demo.db"

def create_demo_database(path: str = DB_PATH) -> None:
    conn = sqlite3.connect(path)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS orders")
    cur.execute("""
        CREATE TABLE orders (
            id INTEGER PRIMARY KEY,
            customer_name TEXT NOT NULL,
            sku TEXT,
            status TEXT,
            tracking TEXT
        )
    """)
    rows = [
        (12345, "John Doe", "WIDGET-A", "shipped", "TRK123456"),
        (12346, "John Doe", "WIDGET-B", "delivered", "TRK123457"),
        (12347, "Jane Smith", "GADGET-1", "processing", None),
        (12348, "Jane Smith", "GADGET-2", "shipped", "TRK123458"),
    ]
    cur.executemany(
        "INSERT INTO orders (id, customer_name, sku, status, tracking) VALUES (?, ?, ?, ?, ?)",
        rows,
    )
    conn.commit()
    conn.close()
    print(f"Demo DB ready: {path}")

create_demo_database()

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")
sql_tools = SQLDatabaseToolkit(db=db, llm=model).get_tools()
search_tool = TavilySearchResults(max_results=2)
tools = [*sql_tools, search_tool]

print("Tools loaded:", len(tools), "total")
for t in tools:
    print(f"  • {t.name}")

Demo DB ready: summarize_middleware_demo.db
Tools loaded: 5 total
  • sql_db_query
  • sql_db_schema
  • sql_db_list_tables
  • sql_db_query_checker
  • tavily_search_results_json


C:\Users\Training\AppData\Local\Temp\ipykernel_28296\4171399265.py:35: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [3]:
def _msg_chars(msg) -> int:
    c = msg.content
    if isinstance(c, str):
        return len(c)
    return len(str(c))


def _print_message_sizes(messages, label: str) -> None:
    total_chars = sum(_msg_chars(m) for m in messages)
    print(f"  {label}: {len(messages)} messages, {total_chars} total chars")
    for idx, msg in enumerate(messages):
        print(f"    [{idx}] type={msg.type} chars={_msg_chars(msg)}")


def simulate_conversation(agent, config, messages_list, title="Conversation"):
    """Run turns and print counts/sizes only (no full message bodies)."""
    print(f"\n🗣️ {title}")
    print("=" * 60)

    for i, message in enumerate(messages_list):
        response = agent.invoke(
            {"messages": [{"role": "user", "content": message}]},
            config,
        )
        msgs = response["messages"]
        last = msgs[-1]
        print(f"\n— Turn {i + 1}/{len(messages_list)} —")
        print(f"  user prompt chars: {len(message)}")
        print(f"  last message type: {last.type}, chars: {_msg_chars(last)}")
        _print_message_sizes(msgs, "checkpoint messages")

In [4]:
# Agent with summarization middleware (same model instance as the SQL toolkit)
summarize_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""You are a helpful assistant with:
    - SQL tools: answer questions about the local SQLite database (orders table). Use read-only SELECT queries; use the toolkit to list tables/schema before querying.
    - Web search: use tavily_search for public, up-to-date facts (shipping carriers, industry news) when the database is not enough.

    Be concise. If you see a conversation summary, use it for continuity across turns.""",
    middleware=[
        SummarizationMiddleware(
            model=ChatOpenAI(model="gpt-4.1-mini"),
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
    checkpointer=InMemorySaver(),
)

print("📝 Summarization-enabled agent created (SQL toolkit + Tavily search)!")
print("🧠 This agent will summarize old messages when the trigger threshold is reached.")
print("⚙️ Settings: trigger every 10 messages, keep last 4 messages verbatim")

📝 Summarization-enabled agent created (SQL toolkit + Tavily search)!
🧠 This agent will summarize old messages when the trigger threshold is reached.
⚙️ Settings: trigger every 10 messages, keep last 4 messages verbatim


In [5]:
config_summarize: RunnableConfig = {"configurable": {"thread_id": "customer_thread_summarize"}}

extended_messages = [
    "Hi — I'm John Doe. What orders do you see for me in the database?",
    "Give me status and tracking for order 12345.",
    "List every order in the database with status processing or shipped.",
    "How does that compare to Jane Smith's orders?",
    "What's the average number of orders per customer in the database?",
    "Search the web: what is UPS Next Day Air in one sentence?",
    "Any public guidance on when 'processing' vs 'shipped' is typical for small parcels?",
    "Which SKUs appear more than once in our orders table?",
    "Summarize my open questions so far in a bullet list.",
    "Run one more SQL check: count rows in orders.",
    "Search briefly for FedEx vs UPS express options (one fact each).",
    "What did we establish about order 12345 earlier in this chat?",
    "Close with a one-line recap of John vs Jane from the DB only.",
]

simulate_conversation(
    summarize_agent,
    config_summarize,
    extended_messages,
    "Memory with SUMMARIZATION Pattern",
)


🗣️ Memory with SUMMARIZATION Pattern

— Turn 1/13 —
  user prompt chars: 65
  last message type: ai, chars: 286
  checkpoint messages: 8 messages, 778 total chars
    [0] type=human chars=65
    [1] type=ai chars=0
    [2] type=tool chars=6
    [3] type=ai chars=0
    [4] type=tool chars=329
    [5] type=ai chars=0
    [6] type=tool chars=92
    [7] type=ai chars=286

— Turn 2/13 —
  user prompt chars: 44
  last message type: ai, chars: 78
  checkpoint messages: 6 messages, 1701 total chars
    [0] type=human chars=1267
    [1] type=ai chars=286
    [2] type=human chars=44
    [3] type=ai chars=0
    [4] type=tool chars=26
    [5] type=ai chars=78

— Turn 3/13 —
  user prompt chars: 67
  last message type: ai, chars: 333
  checkpoint messages: 10 messages, 2272 total chars
    [0] type=human chars=1267
    [1] type=ai chars=286
    [2] type=human chars=44
    [3] type=ai chars=0
    [4] type=tool chars=26
    [5] type=ai chars=78
    [6] type=human chars=67
    [7] type=ai chars=0
   